<a href="https://colab.research.google.com/github/samreenfathima18/guvi-project2/blob/main/project2_stock_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import glob
import yaml
import pandas as pd

def extract_yaml_to_csv(yaml_directory, output_directory):
    """
    Parses nested YAML files structured as lists using explicit keys found in the dataset:
    Ticker, open, high, low, close, volume, date
    """
    os.makedirs(output_directory, exist_ok=True)
    all_records = []

    # Locate all YAML/YML files recursively
    yaml_files = glob.glob(os.path.join(yaml_directory, "**/*.yaml"), recursive=True) + \
                 glob.glob(os.path.join(yaml_directory, "**/*.yml"), recursive=True)

    print(f"Found {len(yaml_files)} YAML files. Beginning extraction...")

    for file_path in yaml_files:
        with open(file_path, 'r') as f:
            try:
                content = yaml.safe_load(f)
                if not content or not isinstance(content, list):
                    continue

                # Directly map your file keys
                for item in content:
                    if not isinstance(item, dict):
                        continue

                    symbol = item.get('Ticker')
                    raw_date = item.get('date')

                    # Ensure minimum essential fields are present
                    if not symbol or not raw_date:
                        continue

                    record = {
                        # Clean date format from '2024-07-18 05:30:00' to just '2024-07-18'
                        'Date': pd.to_datetime(raw_date).date(),
                        'Symbol': str(symbol).strip(),
                        'Open': float(item.get('open', 0)),
                        'High': float(item.get('high', 0)),
                        'Low': float(item.get('low', 0)),
                        'Close': float(item.get('close', 0)),
                        'Volume': int(item.get('volume', 0))
                    }
                    all_records.append(record)

            except Exception as e:
                file_name = os.path.basename(file_path)
                print(f"Error processing file {file_name}: {e}")

    if not all_records:
        print("No records compiled. Please recheck your dataset status.")
        return

    # Create unified master dataframe
    df_master = pd.DataFrame(all_records)

    # Sort chronological details per asset group
    df_master = df_master.sort_values(by=['Symbol', 'Date']).reset_index(drop=True)

    # Save partitioned CSV files by Symbol
    unique_symbols = df_master['Symbol'].unique()
    for symbol in unique_symbols:
        df_symbol = df_master[df_master['Symbol'] == symbol]

        # Strip unexpected characters from filename pathways if any exist
        clean_symbol_name = "".join([c for c in symbol if c.isalnum() or c in ('_', '-')])
        csv_path = os.path.join(output_directory, f"{clean_symbol_name}.csv")
        df_symbol.to_csv(csv_path, index=False)

    print(f"\n✨ Success! Processed {len(all_records):,} total data rows.")
    print(f"📁 Generated {len(unique_symbols)} individual stock symbol CSV files inside your output directory.")

# Run the updated code
extract_yaml_to_csv('/content/drive/MyDrive/project2/project2', '/content/drive/MyDrive/Classroom/stock_csvs')


Found 284 YAML files. Beginning extraction...

✨ Success! Processed 14,200 total data rows.
📁 Generated 50 individual stock symbol CSV files inside your output directory.


In [ ]:
import os
import glob
import pandas as pd
from sqlalchemy import create_engine, Column, String, Date, Float, BigInteger, Integer, ForeignKey
from sqlalchemy.orm import declarative_base

Base = declarative_base()

# Defining Table Schema Structures
class SectorMapping(Base):
    __tablename__ = 'sector_mappings'
    Symbol = Column(String(50), primary_key=True)
    Sector = Column(String(100), nullable=False)

class DailyStockData(Base):
    __tablename__ = 'daily_stock_data'
    id = Column(Integer, primary_key=True, autoincrement=True)
    Date = Column(Date, nullable=False, index=True)
    Symbol = Column(String(50), ForeignKey('sector_mappings.Symbol'), nullable=False, index=True)
    Open = Column(Float)
    High = Column(Float)
    Low = Column(Float)
    Close = Column(Float)
    Volume = Column(BigInteger)

def build_sql_database(db_destination_path, stock_csvs_directory, sector_csv_file):
    """
    Constructs schema tables, aggregates the 50 CSV profiles,
    and seeds a cohesive SQLite analytics relational database.
    """
    # Create SQLite database connection engine string
    engine = create_engine(f"sqlite:///{db_destination_path}")

    print("Initialising relational database schema tables...")
    Base.metadata.drop_all(engine)
    Base.metadata.create_all(engine)

    # Step 1: Ingest and Seeding Sector Maps Table
    print(f"Reading sector definitions from: {os.path.basename(sector_csv_file)}")
    try:
        df_sectors = pd.read_csv(sector_csv_file)
        # Ensure column casings align exactly with our class definitions
        df_sectors.columns = ['Symbol', 'Sector']
        df_sectors['Symbol'] = df_sectors['Symbol'].str.strip()
        df_sectors['Sector'] = df_sectors['Sector'].str.strip()
        df_sectors.to_sql('sector_mappings', engine, if_exists='append', index=False)
        print(f"✓ Seeded {len(df_sectors)} sector maps entries successfully.")
    except Exception as e:
        print(f"❌ Failed processing sectors metadata sheet: {e}")
        return

    # Step 2: Accumulating 50 individual extracted stock profiles
    all_csv_paths = glob.glob(os.path.join(stock_csvs_directory, "*.csv"))
    print(f"Scanning target folders... Found {len(all_csv_paths)} stock data files.")

    combined_rows_list = []
    for file_path in all_csv_paths:
        try:
            df_indv = pd.read_csv(file_path)
            # Ensure proper standard DateTime object parsing
            df_indv['Date'] = pd.to_datetime(df_indv['Date']).dt.date
            df_indv['Symbol'] = df_indv['Symbol'].str.strip()
            combined_rows_list.append(df_indv)
        except Exception as e:
            print(f"Skipping damaged asset entry line item {os.path.basename(file_path)}: {e}")

    if combined_rows_list:
        print("Compiling global analytical framework tables...")
        df_master_stocks = pd.concat(combined_rows_list, ignore_index=True)

        # Enforcing data integrity by filtering items not mapped in sector definitions
        valid_symbols = set(df_sectors['Symbol'])
        df_master_stocks = df_master_stocks[df_master_stocks['Symbol'].isin(valid_symbols)]

        print(f"Bulk loading {len(df_master_stocks):,} trading records into SQL...")
        df_master_stocks.to_sql('daily_stock_data', engine, if_exists='append', index=False)
        print("✨ Database generation sequence complete! File successfully compiled.")
    else:
        print("❌ Database configuration aborted: No stock records located in the directory path.")

# Configuration Target Executions
DB_FILE_PATH = '/content/drive/MyDrive/Classroom/nifty_50_market.db'
CSV_SOURCE_DIR = '/content/drive/MyDrive/Classroom/stock_csvs'

# CHANGE THIS PATH to point to your sector classification CSV spreadsheet file on Drive:
SECTORS_CSV_PATH = '/content/sector.csv'

build_sql_database(DB_FILE_PATH, CSV_SOURCE_DIR, SECTORS_CSV_PATH)


Initialising relational database schema tables...
Reading sector definitions from: sector.csv
✓ Seeded 50 sector maps entries successfully.
Scanning target folders... Found 50 stock data files.
Compiling global analytical framework tables...
Bulk loading 13,632 trading records into SQL...
✨ Database generation sequence complete! File successfully compiled.


In [ ]:
import sqlite3
import pandas as pd

# Connect to your newly built database
conn = sqlite3.connect('/content/drive/MyDrive/Classroom/nifty_50_market.db')

# Print table names and verification records
print("Tables present in DB:", pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)['name'].tolist())
print("\nFirst 3 rows of daily stock ledger:")
display(pd.read_sql("SELECT * FROM daily_stock_data LIMIT 3;", conn))
conn.close()


Tables present in DB: ['sector_mappings', 'daily_stock_data']

First 3 rows of daily stock ledger:


,id,Date,Symbol,Open,High,Low,Close,Volume
0,1,2023-10-03,ADANIENT,2418.00,2424.90,2372.00,2387.25,2019899
1,2,2023-10-04,ADANIENT,2402.20,2502.75,2392.25,2464.95,2857377
2,3,2023-10-05,ADANIENT,2477.95,2486.50,2446.40,2466.35,1132455


In [ ]:
import csv

# Your raw data mapped into a clean dictionary
sector_data = {
    'ADANIENT':'MISCELLANEOUS','ADANIPORTS':'MISCELLANEOUS','APOLLOHOSP':'MISCELLANEOUS',
    'ASIANPAINT':'PAINTS','AXISBANK':'BANKING','BAJAJAUTO':'AUTOMOBILES',
    'BAJFINANCE':'FINANCE','BAJAJFINSV':'FINANCE','BEL':'DEFENCE',
    'BHARTIARTL':'TELECOM','BPCL':'ENERGY','CIPLA':'PHARMACEUTICALS',
    'COALINDIA':'MINING','DRREDDY':'PHARMACEUTICALS','EICHERMOT':'AUTOMOBILES',
    'GRASIM':'TEXTILES','HCLTECH':'SOFTWARE','HDFCBANK':'BANKING',
    'HDFCLIFE':'INSURANCE','HEROMOTOCO':'AUTOMOBILES','HINDALCO':'ALUMINIUM',
    'HINDUNILVR':'FMCG','ICICIBANK':'BANKING','INDUSINDBK':'BANKING',
    'INFY':'SOFTWARE','BRITANNIA':'FMCG','ITC':'FOOD & TOBACCO',
    'JSWSTEEL':'STEEL','KOTAKBANK':'BANKING','LT':'ENGINEERING',
    'M&M':'AUTOMOBILES','MARUTI':'AUTOMOBILES','NESTLEIND':'FOOD & TOBACCO',
    'NTPC':'POWER','ONGC':'ENERGY','POWERGRID':'POWER',
    'RELIANCE':'ENERGY','SBILIFE':'INSURANCE','SBIIN':'BANKING',
    'SHRIRAMFIN':'FINANCE','SUNPHARMA':'PHARMACEUTICALS','TATACONSUM':'FMCG',
    'TATAMOTORS':'AUTOMOBILES','TATASTEEL':'STEEL','TCS':'SOFTWARE',
    'TECHM':'SOFTWARE','TITAN':'RETAILING','TRENT':'RETAILING',
    'ULTRACEMCO':'CEMENT','WIPRO':'SOFTWARE'
}

# Writing to the CSV file
with open('sector.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['Symbol', 'Sector'])  # Headers
    for symbol, sector in sector_data.items():
        writer.writerow([symbol, sector])

print("sector.csv saved successfully!")


sector.csv saved successfully!


In [ ]:
!pip install streamlit matplotlib seaborn sqlalchemy pandas
!npm install -g localtunnel


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 108.9 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
added 22 packages in 2s
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇npm notice
npm notice New major version of npm available! 10.8.2 -> 11.14.1
npm notice Changelog: https://github.com/npm/cli/releases/tag/v11.14.1
npm notice To update run: npm install -g npm@11.14.1
npm notice
⠏

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import sqlite3
import seaborn as sns
import matplotlib.pyplot as plt

# --- Page Configuration ---
st.set_page_config(
    page_title="Nifty 50 Market Analysis Dashboard",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.title("📈 Data-Driven Nifty 50 Performance Dashboard")
st.markdown("---")

# --- Configuration Constants ---
# Replaced with your exact SQLite path from the pipeline step
DB_FILE_PATH = '/content/drive/MyDrive/Classroom/nifty_50_market.db'

# --- Cached Data Loading Layer ---
@st.cache_data
def load_market_data(db_path):
    try:
        conn = sqlite3.connect(db_path)
        query = """
            SELECT
                d.Date,
                d.Symbol,
                d.Open,
                d.High,
                d.Low,
                d.Close,
                d.Volume,
                s.Sector
            FROM daily_stock_data d
            JOIN sector_mappings s ON d.Symbol = s.Symbol
        """
        df = pd.read_sql_query(query, conn)
        conn.close()

        df['Date'] = pd.to_datetime(df['Date'])
        df = df.sort_values(by=['Symbol', 'Date']).reset_index(drop=True)

        df['Prev_Close'] = df.groupby('Symbol')['Close'].shift(1)
        df['Daily_Return'] = (df['Close'] - df['Prev_Close']) / df['Prev_Close']
        df['Daily_Return'] = df['Daily_Return'].fillna(0)
        df['Cum_Return'] = df.groupby('Symbol')['Daily_Return'].transform(lambda x: (1 + x).cumprod() - 1)
        df['Month'] = df['Date'].dt.to_period('M')
        return df
    except Exception as e:
        st.error(f"Failed to extract historical data from SQLite engine: {e}")
        return pd.DataFrame()

df = load_market_data(DB_FILE_PATH)

if df.empty:
    st.warning("⚠️ Database repository empty or unreadable. Please check your DB configuration execution path.")
else:
    summary_metrics = []
    for symbol, grp in df.groupby('Symbol'):
        sorted_grp = grp.sort_values('Date')
        initial_close = sorted_grp['Close'].iloc[0]
        final_close = sorted_grp['Close'].iloc[-1]
        yearly_return = (final_close - initial_close) / initial_close

        summary_metrics.append({
            'Symbol': symbol,
            'Sector': grp['Sector'].iloc[0],
            'Yearly_Return': yearly_return,
            'Avg_Price': grp['Close'].mean(),
            'Avg_Volume': grp['Volume'].mean(),
            'Volatility': grp['Daily_Return'].std()
        })

    metrics_df = pd.DataFrame(summary_metrics)
    metrics_df['Performance_Class'] = np.where(metrics_df['Yearly_Return'] >= 0, 'Green', 'Red')

    st.sidebar.header("🎯 Dashboard Control Filters")
    available_sectors = sorted(metrics_df['Sector'].unique())
    selected_sectors = st.sidebar.multiselect("Filter by Market Sectors", available_sectors, default=available_sectors)

    filtered_symbols_list = sorted(metrics_df[metrics_df['Sector'].isin(selected_sectors)]['Symbol'].unique())
    selected_symbols = st.sidebar.multiselect("Pick Tickers for Comparisons", filtered_symbols_list, default=filtered_symbols_list[:5])

    tab_overview, tab_risk, tab_sector, tab_corr, tab_monthly = st.tabs([
        "📊 Market Summary", "⚡ Volatility & Returns", "🏢 Sector Analytics", "🔗 Price Correlations", "📅 Month-wise Trends"
    ])

    with tab_overview:
        st.header("Overall Market Overview & Rankings")
        c1, c2, c3, c4 = st.columns(4)
        g_count = (metrics_df['Performance_Class'] == 'Green').sum()
        r_count = (metrics_df['Performance_Class'] == 'Red').sum()

        c1.metric("Market Average Close Price", f"₹{metrics_df['Avg_Price'].mean():,.2f}")
        c2.metric("Market Average Volume Traded", f"{int(metrics_df['Avg_Volume'].mean()):,}")
        c3.metric("Green Stocks Count (Gainers) 🟢", f"{g_count} ({g_count/len(metrics_df)*100:.1f}%)")
        c4.metric("Red Stocks Count (Losers) 🔴", f"{r_count} ({r_count/len(metrics_df)*100:.1f}%)")

        st.markdown("### Yearly Stock Performance Rankings")
        col_g, col_l = st.columns(2)
        with col_g:
            st.success("🏆 Top 10 Best-Performing Stocks (Green)")
            top_10_green = metrics_df.sort_values('Yearly_Return', ascending=False)[['Symbol', 'Sector', 'Yearly_Return']].head(10)
            st.dataframe(top_10_green.style.format({'Yearly_Return': '{:.2%}'}), use_container_width=True)
        with col_l:
            st.error("📉 Top 10 Worst-Performing Stocks (Red)")
            top_10_loss = metrics_df.sort_values('Yearly_Return', ascending=True)[['Symbol', 'Sector', 'Yearly_Return']].head(10)
            st.dataframe(top_10_loss.style.format({'Yearly_Return': '{:.2%}'}), use_container_width=True)

    with tab_risk:
        st.header("Risk Performance and Cumulative Growth Trajectories")
        col_v, col_c = st.columns(2)
        with col_v:
            st.subheader("Top 10 Most Volatile Stocks")
            top_v = metrics_df.sort_values('Volatility', ascending=False).head(10)
            fig, ax = plt.subplots(figsize=(6, 4))
            sns.barplot(data=top_v, x='Symbol', y='Volatility', ax=ax, palette='flare')
            plt.xticks(rotation=45)
            plt.ylabel("Standard Deviation of Daily Returns")
            st.pyplot(fig)
        with col_c:
            st.subheader("Cumulative Return Trajectory (Top 5 Performers)")
            top_5_symbols = metrics_df.sort_values('Yearly_Return', ascending=False).head(5)['Symbol'].tolist()
            traj_df = df[df['Symbol'].isin(top_5_symbols)].sort_values('Date')
            fig2, ax2 = plt.subplots(figsize=(6, 4))
            sns.lineplot(data=traj_df, x='Date', y='Cum_Return', hue='Symbol', ax=ax2, linewidth=2)
            plt.xticks(rotation=45)
            plt.ylabel("Cumulative Growth Return %")
            st.pyplot(fig2)

    with tab_sector:
        st.header("Sector-wise Performance Breakdown")
        sector_perf = metrics_df.groupby('Sector')['Yearly_Return'].mean().reset_index().sort_values('Yearly_Return', ascending=False)
        fig3, ax3 = plt.subplots(figsize=(10, 4))
        sns.barplot(data=sector_perf, x='Sector', y='Yearly_Return', ax=ax3, palette='viridis')
        plt.xticks(rotation=45, ha='right')
        plt.ylabel("Average Yearly Return %")
        st.pyplot(fig3)
        st.dataframe(sector_perf.style.format({'Yearly_Return': '{:.2%}'}), use_container_width=True)

    with tab_corr:
        st.header("Stock Closing Price Correlation Heatmap")
        if len(selected_symbols) > 1:
            pivot_df = df[df['Symbol'].isin(selected_symbols)].pivot(index='Date', columns='Symbol', values='Close')
            corr_matrix = pivot_df.corr(method='pearson')
            fig4, ax4 = plt.subplots(figsize=(8, 6))
            sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", ax=ax4, vmin=-1, vmax=1, center=0)
            st.pyplot(fig4)
        else:
            st.warning("⚠️ Please select at least two stock tickers in the sidebar filter to view the correlation matrix.")

    with tab_monthly:
        st.header("Granular Month-by-Month Trends")
        monthly_bounds = df.sort_values(['Symbol', 'Date']).groupby(['Symbol', 'Month'])['Close'].agg(['first', 'last']).reset_index()
        monthly_bounds['Monthly_Return'] = (monthly_bounds['last'] - monthly_bounds['first']) / monthly_bounds['first']

        available_months = sorted(monthly_bounds['Month'].unique())
        selected_month = st.selectbox("Select Target Analysis Month", available_months)

        month_data = monthly_bounds[monthly_bounds['Month'] == selected_month]
        m_gainers = month_data.sort_values('Monthly_Return', ascending=False).head(5)
        m_losers = month_data.sort_values('Monthly_Return', ascending=True).head(5)

        mg_col, ml_col = st.columns(2)
        with mg_col:
            st.success(f"🟢 Top 5 Monthly Gainers ({selected_month})")
            fig5, ax5 = plt.subplots(figsize=(5, 3))
            sns.barplot(data=m_gainers, x='Symbol', y='Monthly_Return', ax=ax5, palette='Greens_r')
            st.pyplot(fig5)
        with ml_col:
            st.error(f"🔴 Top 5 Monthly Losers ({selected_month})")
            fig6, ax6 = plt.subplots(figsize=(5, 3))
            sns.barplot(data=m_losers, x='Symbol', y='Monthly_Return', ax=ax6, palette='Reds_r')
            st.pyplot(fig6)


Writing app.py


In [ ]:
!curl ://icanhazip.com
!streamlit run app.py & npx localtunnel --port 8501


curl: (3) URL using bad/illegal format or missing URL
⠙⠹⠸⠼⠴⠦

your url is: https://quick-tips-sneeze.loca.lt
2026-05-20 12:17:11.021 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.204.116.216:8501

2026-05-20 12:18:03.789 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-05-20 12:18:03.830 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
/content/app.py:120: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and se